# ScamSense — 01: Environment Setup & Data Download

**Purpose:** Mount Google Drive, clone the ScamSense GitHub repo, create the project folder structure, install all dependencies, configure Kaggle credentials, and download all raw datasets.

**Run this notebook once per Colab session before running any other notebook.**

| Step | What it does |
|---|---|
| Cell 1 | Mount Drive + clone/pull GitHub repo |
| Cell 2 | Create project folder structure |
| Cell 3 | Install all Python dependencies |
| Cell 4 | Configure Kaggle API credentials |
| Cell 5 | Download all raw datasets |
| Cell 6 | Verify all datasets are present |
| Cell 7 | Upload synthetic scam data (manual step) |

In [ ]:
# =============================================================================
# CELL 1: Mount Google Drive and connect to GitHub repo
# -----------------------------------------------------------------------------
# - Mounts Drive so all files persist across Colab sessions
# - Clones the ScamSense repo into Drive on first run
# - Pulls latest changes on subsequent runs
# - Sets working directory to the repo root for all subsequent cells
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_URL = "https://github.com/Bhoovika/ScamSense.git"
REPO_DIR = "/content/drive/MyDrive/ScamSense"

if not os.path.exists(REPO_DIR):
    print("Cloning repo for the first time...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already exists — pulling latest changes...")
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

Mounted at /content/drive
Cloning repo for the first time...
Cloning into '/content/drive/MyDrive/ScamSense'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
Working directory: /content/drive/MyDrive/ScamSense


In [ ]:
# =============================================================================
# CELL 2: Create ScamSense project folder structure
# -----------------------------------------------------------------------------
# Creates all required directories if they don't already exist.
# safe to re-run — os.makedirs with exist_ok=True will not overwrite.
#
# data/raw/         — original downloaded datasets, never modified
# data/processed/   — cleaned and merged datasets ready for modelling
# notebooks/        — all Jupyter notebooks
# src/              — reusable Python modules (agents, utils, etc.)
# models/           — saved model weights and MLflow artifacts
# reports/figures/  — EDA plots and evaluation charts
# =============================================================================
folders = [
    "data/raw",
    "data/processed",
    "notebooks",
    "src",
    "models/checkpoints",
    "reports/figures",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✓ {folder}")

print("\nFolder structure ready.")

✓ data/raw
✓ data/processed
✓ notebooks
✓ src
✓ models/checkpoints
✓ reports/figures

Folder structure ready.


In [ ]:
# =============================================================================
# CELL 3: Install all Python dependencies
# -----------------------------------------------------------------------------
# Installs all packages required across the full ScamSense pipeline:
#   transformers      — XLM-RoBERTa, mBERT, NLLB-200 models
#   datasets          — HuggingFace Wikipedia multilingual data
#   torch             — model training and inference
#   scikit-learn      — evaluation metrics, train/test split
#   pandas / numpy    — data manipulation
#   mlflow            — experiment tracking
#   evidently         — data drift detection
#   langchain*        — LangGraph multi-agent pipeline
#   chromadb          — vector store for RAG pipeline
#   python-telegram-bot — Telegram bot interface
#   kaggle            — Kaggle dataset downloads
#   huggingface_hub   — model and dataset uploads
#   matplotlib/seaborn — EDA visualisations
#   sentence-transformers — embeddings for RAG
#   accelerate        — multi-GPU / mixed precision training
# =============================================================================
!pip install -q \
    transformers==4.40.0 \
    datasets \
    torch \
    scikit-learn \
    pandas numpy \
    mlflow \
    evidently \
    langchain langchain-community \
    chromadb \
    python-telegram-bot \
    kaggle \
    huggingface_hub \
    matplotlib seaborn \
    sentence-transformers \
    accelerate

print("All packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.4/745.

In [ ]:
# =============================================================================
# CELL 4A: Kaggle API setup — FIRST TIME ONLY
# -----------------------------------------------------------------------------
# Run this cell only on the very first setup.
# It prompts you to upload kaggle.json from your local computer,
# places it where the Kaggle CLI expects it (/root/.config/kaggle/),
# and saves a copy to Drive so future sessions skip the upload step.
#
# To get kaggle.json: kaggle.com -> Account -> API -> Create New Token
# WARNING: never commit kaggle.json to GitHub.
# =============================================================================
from google.colab import files
import shutil

uploaded = files.upload()  # select kaggle.json from your local computer

os.makedirs("/root/.config/kaggle", exist_ok=True)
shutil.copy("kaggle.json", "/root/.config/kaggle/kaggle.json")
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)

# Save to Drive so future sessions can load it without re-uploading
shutil.copy("kaggle.json", "/content/drive/MyDrive/kaggle.json")

print("Kaggle credentials configured and saved to Drive.")

Saving kaggle.json to kaggle.json
Kaggle credentials configured and saved to Drive.


In [ ]:
# =============================================================================
# CELL 4B: Kaggle API setup — SUBSEQUENT SESSIONS
# -----------------------------------------------------------------------------
# Run this instead of 4A after the first setup.
# Loads kaggle.json from Drive — no upload needed.
# =============================================================================
import shutil

os.makedirs("/root/.config/kaggle", exist_ok=True)
shutil.copy("/content/drive/MyDrive/kaggle.json", "/root/.config/kaggle/kaggle.json")
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)

print("Kaggle credentials loaded from Drive.")

Kaggle credentials loaded from Drive.


In [ ]:
# =============================================================================
# CELL 5: Download all raw datasets
# -----------------------------------------------------------------------------
# Downloads 5 datasets into data/raw/ subfolders:
#
# 1. Phishing Email Dataset (Kaggle)     — 82,486 emails, label 0/1
# 2. NUS SMS Corpus (Kaggle)             — 67,093 SMS, Singlish/Mandarin ham
# 3. Real/Fake Job Postings (Kaggle)     — 17,880 postings, label 0/1
# 4. UCI SMS Spam Collection (UCI repo)  — 5,574 SMS, label spam/ham
# 5. Wikipedia multilingual (HuggingFace)— 5,000 paragraphs each for MS/TA/ZH ham
#
# Synthetic scam data is uploaded separately in Cell 7.
# =============================================================================
from datasets import load_dataset

RAW = "data/raw"

# 1. Phishing Email Dataset
print("Downloading Phishing Email Dataset...")
!kaggle datasets download -d naserabdullahalam/phishing-email-dataset -p {RAW}/phishing --unzip

# 2. NUS SMS Corpus (Singlish + Mandarin ham source)
print("Downloading NUS SMS Corpus...")
!kaggle datasets download -d rtatman/the-national-university-of-singapore-sms-corpus -p {RAW}/nus_sms --unzip

# 3. Real/Fake Job Postings
print("Downloading Real/Fake Job Postings...")
!kaggle datasets download -d shivamb/real-or-fake-fake-jobposting-prediction -p {RAW}/job_postings --unzip

# 4. UCI SMS Spam Collection
print("Downloading UCI SMS Spam Collection...")
os.makedirs(f"{RAW}/uci_sms", exist_ok=True)
!wget -q "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip" \
     -O {RAW}/uci_sms/sms_spam.zip
!unzip -o {RAW}/uci_sms/sms_spam.zip -d {RAW}/uci_sms/

# 5. Wikipedia multilingual paragraphs (Malay, Tamil, Mandarin — used as ham label=0)
print("Downloading Wikipedia multilingual...")
os.makedirs(f"{RAW}/wikipedia", exist_ok=True)
for lang_code, lang_name in [("ms", "Malay"), ("ta", "Tamil"), ("zh", "Mandarin")]:
    print(f"  Fetching Wikipedia {lang_name} (first 5,000 paragraphs)...")
    ds = load_dataset(
        "wikimedia/wikipedia",
        f"20231101.{lang_code}",
        split="train[:5000]",
        trust_remote_code=True,
    )
    ds.to_csv(f"{RAW}/wikipedia/wiki_{lang_code}.csv", index=False)
    print(f"  ✓ {len(ds)} rows saved to wiki_{lang_code}.csv")

print("\nAll downloads complete.")

Dataset URL: https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset
License(s): CC-BY-SA-4.0
100% 77.1M/77.1M [00:00<00:00, 89.2MB/s]

Dataset URL: https://www.kaggle.com/datasets/rtatman/the-national-university-of-singapore-sms-corpus
License(s): other
100% 3.61M/3.61M [00:00<00:00, 152MB/s]

Dataset URL: https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobposting-prediction
License(s): CC0-1.0
100% 16.1M/16.1M [00:00<00:00, 97.5MB/s]

Archive:  data/raw/uci_sms/sms_spam.zip
  inflating: data/raw/uci_sms/SMSSpamCollection  
  inflating: data/raw/uci_sms/readme  


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikimedia/wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikimedia/wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  Fetching Wikipedia Malay (first 5,000 paragraphs)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

...

20231101.zh/train-00004-of-00006.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

20231101.zh/train-00005-of-00006.parquet:   0%|          | 0.00/225M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1384748 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

  ✓ 5000 rows saved to wiki_zh.csv

All downloads complete.


In [ ]:
# =============================================================================
# CELL 6: Verify all datasets are present
# -----------------------------------------------------------------------------
# Checks that every expected raw data file or folder exists.
# Prints size/file count for each. Fix any ❌ before proceeding.
# =============================================================================
checks = {
    "Phishing Email":        "data/raw/phishing",
    "NUS SMS Corpus":        "data/raw/nus_sms",
    "Real/Fake Job Postings":"data/raw/job_postings",
    "UCI SMS Spam":          "data/raw/uci_sms",
    "Wikipedia Malay":       "data/raw/wikipedia/wiki_ms.csv",
    "Wikipedia Tamil":       "data/raw/wikipedia/wiki_ta.csv",
    "Wikipedia Mandarin":    "data/raw/wikipedia/wiki_zh.csv",
    "Synthetic Scam Data":   "data/raw/synthetic/scamsense_synthetic_scam_only.csv",
}

all_good = True
for name, path in checks.items():
    exists = os.path.exists(path)
    if not exists:
        all_good = False
        print(f"❌ {name}: NOT FOUND at {path}")
    elif os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"✅ {name}: {path} ({size_mb:.1f} MB)")
    else:
        n_files = len(os.listdir(path))
        print(f"✅ {name}: {path} ({n_files} files)")

print()
if all_good:
    print("All datasets present — ready to proceed to notebook 02.")
else:
    print("Fix missing datasets above before continuing.")

✅ Phishing Email: data/raw/phishing (7 files)
✅ NUS SMS Corpus: data/raw/nus_sms (2 files)
✅ Real/Fake Job Postings: data/raw/job_postings (1 files)
✅ UCI SMS Spam: data/raw/uci_sms (3 files)
✅ Wikipedia Malay: data/raw/wikipedia/wiki_ms.csv (21.8 MB)
✅ Wikipedia Tamil: data/raw/wikipedia/wiki_ta.csv (53.2 MB)
✅ Wikipedia Mandarin: data/raw/wikipedia/wiki_zh.csv (54.2 MB)
✅ Synthetic Scam Data: data/raw/synthetic/scamsense_synthetic_scam_only.csv (2.6 MB)

All datasets present — ready to proceed to notebook 02.


In [ ]:
# =============================================================================
# CELL 7: Upload synthetic scam data (manual step)
# -----------------------------------------------------------------------------
# The synthetic dataset was generated by notebook 02 and saved locally.
# Upload scamsense_synthetic_scam_only.csv from your computer when prompted.
# This cell moves it to the correct location: data/raw/synthetic/
#
# Skip this cell if the file is already present (Cell 6 showed ✅ for it).
# =============================================================================
from google.colab import files
import shutil

os.makedirs("data/raw/synthetic", exist_ok=True)
uploaded = files.upload()  # select scamsense_synthetic_scam_only.csv

shutil.move(
    "scamsense_synthetic_scam_only.csv",
    "data/raw/synthetic/scamsense_synthetic_scam_only.csv",
)
print("Synthetic data uploaded.")

# Sanity check
import pandas as pd
df = pd.read_csv("data/raw/synthetic/scamsense_synthetic_scam_only.csv")
print(f"Shape: {df.shape}")
print(f"Languages:\n{df['language'].value_counts()}")
print(f"Scam types:\n{df['scam_type'].value_counts()}")
print(f"Labels (all should be 1): {df['label'].unique()}")

Saving scamsense_synthetic_scam_only.csv to scamsense_synthetic_scam_only.csv
Synthetic data uploaded.
Shape: (20482, 4)
Languages:
language
en          5000
ta          4761
zh          3893
singlish    3704
ms          3124
Name: count, dtype: int64
Scam types:
scam_type
job_scam                    2249
phishing                    1906
fake_friend                 1833
rental                      1830
investment                  1792
parcel_delivery             1735
charity                     1694
prize                       1673
loan                        1657
bank_impersonation          1628
ecommerce                   1539
government_impersonation     946
Name: count, dtype: int64
Labels (all should be 1): [1]
